# 环境配置

## 安装包管理工具

In [1]:
%pip install uv

!uv --version

/github.com/sammyne/Deep-Learning-with-Python-2ed-cn/chapter14/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
uv 0.7.19


## 安装依赖

In [ ]:
!uv pip install datasets==3.6.0 transformers==4.53.1

⠙                                                                                 × No solution found when resolving dependencies:
  ╰─▶ Because there is no version of tokenizer==0.21.3 and you require
      tokenizer==0.21.3, we can conclude that your requirements are
      unsatisfiable.


In [2]:
!uv add tokenizers==0.21.2

Resolved 48 packages in 0.57ms
Audited 42 packages in 0.02ms


### 11.2.3 建立词表索引

In [ ]:
vocabulary = {}
for text in dataset:
  text = standardize(text)
  tokens = tokenize(text)
  for token in tokens:
    if token not in vocabulary:
      vocabulary[token] = len(vocabulary)

### 11.2.4 使用 TextVectorization 层

In [6]:
import string

class Vectorizer:
    def standardize(self, text):
        text = text.lower()
        return "".join(char for char in text if char not in string.punctuation)

    def tokenize(self, text):
        text = self.standardize(text)
        return text.split()

    def make_vocabulary(self, dataset):
        self.vocabulary = {"": 0, "[UNK]": 1}
        for text in dataset:
            text = self.standardize(text)
            tokens = self.tokenize(text)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)
        self.inverse_vocabulary = dict(
            (v, k) for k, v in self.vocabulary.items())

    def encode(self, text):
        text = self.standardize(text)
        tokens = self.tokenize(text)
        return [self.vocabulary.get(token, 1) for token in tokens]

    def decode(self, int_sequence):
        return " ".join(
            self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

vectorizer = Vectorizer()
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]
vectorizer.make_vocabulary(dataset)

In [7]:
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = vectorizer.encode(test_sentence)
print(encoded_sentence)

[2, 3, 5, 7, 1, 5, 6]


In [8]:
decoded_sentence = vectorizer.decode(encoded_sentence)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


In [2]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Punctuation,Sequence,Whitespace

def new_tokenizer():
  tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
  tokenizer.enable_padding()
  tokenizer.normalizer = Lowercase()
  tokenizer.pre_tokenizer = Sequence(
      [
          Whitespace(),
          Punctuation(behavior='removed')
      ]
  )
  return tokenizer

In [3]:
from tokenizers.trainers import WordLevelTrainer

trainer = WordLevelTrainer(special_tokens=["[UNK]"])

tokenizer = new_tokenizer()

dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]
tokenizer.train_from_iterator(dataset, trainer=trainer, length=len(dataset))

#### 展示词汇表

In [4]:
tokenizer.get_vocab()

{'i': 6,
 'poppy': 7,
 'then': 9,
 '[UNK]': 0,
 'rewrite': 8,
 'write': 10,
 'blooms': 5,
 'erase': 1,
 'a': 2,
 'and': 4,
 'again': 3}

In [5]:
vocabulary = tokenizer.get_vocab()
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = tokenizer.encode(test_sentence)
print(encoded_sentence)

Encoding(num_tokens=7, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [6]:
inverse_vocab = {v: k for k, v in tokenizer.get_vocab().items()}
decoded_sentence = " ".join(inverse_vocab[int(i)] for i in encoded_sentence.ids)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


## 11.3 表示单词组的两种方法：集合和序列

### 11.3.1 准备 IMDB 影评数据

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  3388k      0  0:00:24  0:00:24 --:--:-- 3715kM    0     0  3072k      0  0:00:26  0:00:14  0:00:12 4036k


In [2]:
!tar -xf aclImdb_v1.tar.gz

In [3]:
!rm -r aclImdb/train/unsup

In [4]:
!cat aclImdb/train/pos/4077_10.txt

I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drink and food in reach so you don't have to stop the film.<br /><br />Enjoy

In [6]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)

In [5]:
!uv pip install datasets==3.6.0

Resolved 32 packages in 815ms                                        
⠙ Preparing packages... (0/14)                                                  
⠙ Preparing packages... (0/14)-------------     0 B/45.59 KiB           
⠙ Preparing packages... (0/14)-------------     0 B/45.59 KiB           
aiohappyeyeballs     ------------------------------     0 B/14.91 KiB
⠙ Preparing packages... (0/14)-------------     0 B/45.59 KiB           
aiohappyeyeballs     ------------------------------     0 B/14.91 KiB
⠙ Preparing packages... (0/14)-------------     0 B/45.59 KiB           
aiohappyeyeballs     ------------------------------     0 B/14.91 KiB
⠙ Preparing packages... (0/14)-------------     0 B/45.59 KiB           
aiosignal            ------------------------------     0 B/7.31 KiB
aiohappyeyeballs     ------------------------------     0 B/14.91 KiB
⠙ Preparing packages... (0/14)-------------     0 B/45.59 KiB           
aiosignal            ------------------------------     0 B/7.

In [ ]:
import datasets

def text_dataset_from_dir(dir, classes):
    data = []
    for c in classes:
        d = datasets.load_dataset("text", data_dir=f"{dir}/{c}")["train"]
        data.append(d.map(lambda x: {"label": c}))
    
    return datasets.concatenate_datasets(data)
    

batch_size = 32

ds = text_dataset_from_dir("aclImdb/world", ["pos", "neg"])
print(ds[:3])

#train_ds = datasets.load_dataset(path="text", data_dir="aclImdb/world", data_files="**/*.txt")

data_files = {
    "train": "aclImdb/world/**/*.txt",
    "world": "aclImdb/world/**/*.txt",
}

train_ds = datasets.load_dataset("text", data_files=data_files)

{'text': ["If you like adult comedy cartoons, like South Park, then this is nearly a similar format about the small adventures of three teenage girls at Bromwell High. Keisha, Natella and Latrina have given exploding sweets and behaved like bitches, I think Keisha is a good leader. There are also small stories going on with the teachers of the school. There's the idiotic principal, Mr. Bip, the nervous Maths teacher and many others. The cast is also fantastic, Lenny Henry's Gina Yashere, EastEnders Chrissie Watts, Tracy-Ann Oberman, Smack The Pony's Doon Mackichan, Dead Ringers' Mark Perry and Blunder's Nina Conti. I didn't know this came from Canada, but it is very good. Very good!", "If you like adult comedy cartoons, like South Park, then this is nearly a similar format about the small adventures of three teenage girls at Bromwell High. Keisha, Natella and Latrina have given exploding sweets and behaved like bitches, I think Keisha is a good leader. There are also small stories goin

In [14]:
train_ds
print(train_ds)

ds = train_ds["train"].map(lambda x: {"label": "pos"})
ds
#train_ds["text"]

# train_ds.map()

#train_ds.features

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2
    })
    world: Dataset({
        features: ['text'],
        num_rows: 2
    })
})


Dataset({
    features: ['text', 'label'],
    num_rows: 2
})